# FERRUS ANIMUS — Notebook Principal
**POUR L'EMPEROR. POUR LA FLOTTE FERRUS.**

**Depot GitHub :** [kioka8877-ux/FLOTTE-FERRUS](https://github.com/kioka8877-ux/FLOTTE-FERRUS/tree/main)

---

## Pipeline
```
IN/ (FBX DeepMotion)
     |
  [CELL 4 - INTEL]   pre_parse_fbx (bpy) → XML → Claude API → intel_skeleton.json
  [CELL 5 - GEMINI]  injection optionnelle intel_vision.json (Gemini Chat)
  [CELL 6 - MERGE]   intel_skeleton + intel_vision → plan_corrections.json
     |
  [CELL 7 - EXEC]    smooth → hips → foot → mask → camera
     |
  [CELL 8 - OUTPUT]  retarget DeepMotion → R15
     |
OUT/ (FBX R15 propres + rapport.json)
```

## Ordre d'execution
1. **CELL 2 - SETUP** — Executer une seule fois par session
2. **CELL 3 - CONFIG** — Verifier les chemins et la cle API
3. **CELL 4 - SCAN** — Deposer les FBX dans `IN/` puis scanner
4. **CELL 5 - INTEL** — Pre-parse + analyse IA
5. **CELL 6 - GEMINI** — Optionnel : injection intel_vision.json
6. **CELL 7 - MERGE** — Produit plan_corrections.json
7. **CELL 8 - EXEC** — 5 operations de correction
8. **CELL 9 - OUTPUT** — Retargeting R15
9. **CELL 10 - RAPPORT** — Rapport final

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — SETUP
# Montage Drive, chemins, installation Blender, helpers
# Executer UNE SEULE FOIS au debut de chaque session Colab
# ═══════════════════════════════════════════════════════════════
import os, sys, subprocess, json, time, shutil

# ── Montage Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Sync codebase GitHub → Drive ─────────────────────────────────
GH_URL   = 'https://github.com/kioka8877-ux/FLOTTE-FERRUS.git'
DRIVE_GH = '/content/drive/MyDrive/FLOTTE-FERRUS'

if not os.path.exists(os.path.join(DRIVE_GH, '.git')):
    print('[SETUP] Premiere session — clone depuis GitHub vers Drive...')
    subprocess.run(['git', 'clone', GH_URL, DRIVE_GH], check=True)
    print('[SETUP] Clone OK')
else:
    print('[SETUP] Git pull depuis GitHub...')
    r = subprocess.run(['git', '-C', DRIVE_GH, 'pull', 'origin', 'main'],
                       capture_output=True, text=True)
    print(r.stdout.strip() or '[SETUP] Already up to date')
    print('[SETUP] Pull OK')

# ── Chemins principaux (adapter FERRUS_ROOT si besoin) ───────────
FERRUS_ROOT = '/content/drive/MyDrive/FLOTTE-FERRUS/01_FERRUS_ANIMUS'
CODEBASE    = os.path.join(FERRUS_ROOT, 'codebase')
INTEL_DIR   = os.path.join(CODEBASE, 'INTEL')
EXEC_DIR    = os.path.join(CODEBASE, 'EXEC', 'operations')
OUTPUT_DIR  = os.path.join(CODEBASE, 'OUTPUT')
IN_DIR      = os.path.join(FERRUS_ROOT, 'IN')
OUT_DIR     = os.path.join(FERRUS_ROOT, 'OUT')
TOOLS_DIR   = '/content/drive/MyDrive/tools'
TMP_DIR     = '/content/tmp_ferrus'

GEMINI_IN   = os.path.join(FERRUS_ROOT, 'GEMINI_IN')
CLAUDE_IN   = os.path.join(FERRUS_ROOT, 'CLAUDE_IN')

os.makedirs(TMP_DIR,   exist_ok=True)
os.makedirs(OUT_DIR,   exist_ok=True)
os.makedirs(IN_DIR,    exist_ok=True)
os.makedirs(GEMINI_IN, exist_ok=True)
os.makedirs(CLAUDE_IN, exist_ok=True)

# ── Installation / verification Blender ──────────────────────────
BLENDER_DRIVE_DIR = os.path.join(TOOLS_DIR, 'blender-4.2')
BLENDER_BIN       = os.path.join(BLENDER_DRIVE_DIR, 'blender')

if not os.path.exists(BLENDER_BIN):
    print('[SETUP] Blender introuvable dans Drive — telechargement (une seule fois)...')
    os.makedirs(TOOLS_DIR, exist_ok=True)
    DL_MIRRORS = [
        'https://download.blender.org/release/Blender4.2/blender-4.2.0-linux-x64.tar.xz',
        'https://ftp.nluug.nl/pub/graphics/blender/release/Blender4.2/blender-4.2.0-linux-x64.tar.xz',
        'https://mirrors.dotsrc.org/blender/release/Blender4.2/blender-4.2.0-linux-x64.tar.xz',
    ]
    DL_OUT     = '/tmp/blender.tar.xz'
    downloaded = False
    for mirror in DL_MIRRORS:
        host = mirror.split('/')[2]
        print(f'[SETUP] Miroir : {host} ...')
        r = subprocess.run(
            ['wget', '-q', '--show-progress', '--tries=2', '--timeout=60',
             '-O', DL_OUT, mirror]
        )
        if r.returncode == 0:
            downloaded = True
            print(f'[SETUP] Telechargement OK depuis {host}')
            break
        print(f'[SETUP] Echec (code {r.returncode}) — miroir suivant...')
    if not downloaded:
        raise RuntimeError('[SETUP] Tous les miroirs Blender ont echoue — verifiez la connexion reseau Colab')
    subprocess.run(['tar', 'xf', DL_OUT, '-C', '/tmp/'], check=True)
    extracted = next(e for e in os.listdir('/tmp/') if e.startswith('blender-4.2'))
    shutil.copytree(os.path.join('/tmp', extracted), BLENDER_DRIVE_DIR)
    print(f'[SETUP] Blender installe : {BLENDER_BIN}')
else:
    print(f'[SETUP] Blender OK : {BLENDER_BIN}')

# ── Permissions executables (Drive ne persiste pas les bits Unix) ────
import stat as _stat
os.chmod(BLENDER_BIN, _stat.S_IRWXU | _stat.S_IRGRP | _stat.S_IXGRP | _stat.S_IROTH | _stat.S_IXOTH)

# ── Dependances Python Colab ──────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests'], check=True)

# ── Helper : wrapper Blender pour pre_parse_fbx ───────────────────
PRE_PARSE_WRAPPER = os.path.join(TMP_DIR, 'blender_pre_parse.py')
with open(PRE_PARSE_WRAPPER, 'w', encoding='utf-8') as _f:
    _f.write('import sys\n')
    _f.write(f'sys.path.insert(0, r"{INTEL_DIR}")\n')
    _f.write('from pre_parse_fbx import extract_fbx_metadata\n')
    _f.write('argv = sys.argv[sys.argv.index("--") + 1:] if "--" in sys.argv else sys.argv[1:]\n')
    _f.write('fbx_path, xml_out = argv[0], argv[1]\n')
    _f.write('xml = extract_fbx_metadata(fbx_path)\n')
    _f.write('with open(xml_out, "w", encoding="utf-8") as fout:\n')
    _f.write('    fout.write(xml)\n')
    _f.write('print("[pre_parse] XML sauvegarde :", xml_out)\n')

# ── Helper : module intel API (intel_skeleton sans import bpy) ────
_intel_src_path = os.path.join(INTEL_DIR, 'intel_skeleton.py')
_intel_api_path = os.path.join(TMP_DIR, 'ferrus_intel_api.py')
with open(_intel_src_path, 'r', encoding='utf-8') as _f:
    _src = _f.read()
_src = _src.replace('from pre_parse_fbx import extract_fbx_metadata\n', '')
with open(_intel_api_path, 'w', encoding='utf-8') as _f:
    _f.write(_src)

if TMP_DIR not in sys.path:
    sys.path.insert(0, TMP_DIR)

print()
print('[SETUP] ================================================')
print('[SETUP] FERRUS ANIMUS pret pour la session')
print(f'[SETUP] IN        : {IN_DIR}')
print(f'[SETUP] OUT       : {OUT_DIR}')
print(f'[SETUP] GEMINI_IN : {GEMINI_IN}')
print(f'[SETUP] CLAUDE_IN : {CLAUDE_IN}')
print(f'[SETUP] TMP       : {TMP_DIR}')
print('[SETUP] ================================================')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — CONFIG
# Charge la cle API depuis les Secrets Colab
# Ajouter AI_GATEWAY_API_KEY dans Secrets (icone cle a gauche)
# ═══════════════════════════════════════════════════════════════

try:
    from google.colab import userdata
    AI_GATEWAY_API_KEY = userdata.get('AI_GATEWAY_API_KEY')
    print('[CONFIG] AI_GATEWAY_API_KEY charge depuis les Secrets Colab')
except Exception:
    AI_GATEWAY_API_KEY = None
    print('[CONFIG] AI_GATEWAY_API_KEY absent — mode fallback statique actif')
    print('[CONFIG] Conseil : ajoutez la cle dans Secrets Colab pour activer Claude API')

print(f'[CONFIG] Mode INTEL : {"API Gateway (Claude)" if AI_GATEWAY_API_KEY else "Fallback statique"}')
print(f'[CONFIG] Blender    : {BLENDER_BIN}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — SCAN
# Scanne IN/ et genere le manifest des personnes a traiter
# Deposer les FBX DeepMotion dans IN/ avant d'executer
# ═══════════════════════════════════════════════════════════════
import glob as _glob

fbx_files = sorted(
    _glob.glob(os.path.join(IN_DIR, '*.fbx')) +
    _glob.glob(os.path.join(IN_DIR, '*.FBX'))
)

if not fbx_files:
    print(f'[SCAN] AUCUN FBX dans {IN_DIR}')
    print('  → Deposez vos fichiers FBX DeepMotion dans IN/ puis relancez cette cellule')
else:
    manifest = {'personnes': []}
    print(f'[SCAN] {len(fbx_files)} FBX detecte(s) :')
    for i, fbx_path in enumerate(fbx_files, 1):
        stem     = os.path.splitext(os.path.basename(fbx_path))[0]
        pid      = f'P{i}'
        taille   = round(os.path.getsize(fbx_path) / (1024 * 1024), 2)
        manifest['personnes'].append({
            'person_id': pid,
            'fbx_input': fbx_path,
            'fbx_stem':  stem,
            'taille_mb': taille,
        })
        print(f'  {pid} : {os.path.basename(fbx_path)} ({taille} MB)')

    manifest_path = os.path.join(TMP_DIR, 'manifest.json')
    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print(f'\n[SCAN] Manifest sauvegarde : {manifest_path}')
    print('[SCAN] Validez ce manifest avant de continuer')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — INTEL
# Pour chaque FBX :
#   Step 1 : pre_parse via Blender → XML
#   Step 2 : analyse Claude API (ou fallback statique) → intel_skeleton.json
# ═══════════════════════════════════════════════════════════════
from ferrus_intel_api import analyze_fbx_with_gateway, _static_fallback

# Charger le manifest
manifest_path = os.path.join(TMP_DIR, 'manifest.json')
assert os.path.exists(manifest_path), 'Executer CELL 4 - SCAN en premier'
with open(manifest_path, encoding='utf-8') as f:
    manifest = json.load(f)

intel_results  = {}  # {pid: analyse_dict}
t_intel_start  = time.time()

for person in manifest['personnes']:
    fbx_path = person['fbx_input']
    stem     = person['fbx_stem']
    pid      = person['person_id']

    cache_path = os.path.join(TMP_DIR, f'intel_skeleton_{stem}.json')

    # ── Check CLAUDE_IN (mode Chat sans API, persistant sur Drive) ──
    claude_in_file = os.path.join(CLAUDE_IN, f'intel_skeleton_{stem}.json')
    if os.path.exists(claude_in_file) and not os.path.exists(cache_path):
        print(f'[INTEL] {pid} — Fichier Claude Chat detecte dans CLAUDE_IN/')
        shutil.copy2(claude_in_file, cache_path)
        print(f'[INTEL] {pid} — Copie vers cache : {os.path.basename(cache_path)}')

    # ── Cache hit ────────────────────────────────────────────────
    if os.path.exists(cache_path):
        print(f'[INTEL] {pid} — Cache hit : {stem}')
        with open(cache_path, encoding='utf-8') as f:
            intel_results[pid] = json.load(f)
        continue

    print(f'\n[INTEL] {pid} — Traitement : {stem}')

    # ── Step 1 : pre-parse via Blender ───────────────────────────
    xml_path = os.path.join(TMP_DIR, f'pre_parse_{stem}.xml')
    print(f'  [pre_parse] Lancement Blender...')
    t0 = time.time()
    proc = subprocess.run(
        [BLENDER_BIN, '--background', '--python', PRE_PARSE_WRAPPER,
         '--', fbx_path, xml_path],
        capture_output=True, text=True, timeout=180
    )
    elapsed = round(time.time() - t0, 1)

    if proc.returncode != 0 or not os.path.exists(xml_path):
        print(f'  [pre_parse] ERREUR (code {proc.returncode}) en {elapsed}s')
        print(proc.stderr[-800:] if proc.stderr else '(pas de stderr)')
        continue
    print(f'  [pre_parse] OK ({elapsed}s)')

    with open(xml_path, encoding='utf-8') as f:
        fbx_xml = f.read()

    # ── Step 2 : analyse IA ou fallback ──────────────────────────
    t0 = time.time()
    if AI_GATEWAY_API_KEY:
        print(f'  [intel] Appel Claude API...')
        try:
            analysis = analyze_fbx_with_gateway(fbx_path, fbx_xml, AI_GATEWAY_API_KEY)
        except Exception as e:
            print(f'  [intel] Erreur API : {e} — fallback statique')
            analysis = _static_fallback(fbx_path, fbx_xml)
    else:
        print(f'  [intel] Fallback statique (pas de cle API)')
        analysis = _static_fallback(fbx_path, fbx_xml)

    elapsed = round(time.time() - t0, 1)
    score       = analysis.get('qualite_fcurves', {}).get('score_global', '?')
    corrections = analysis.get('corrections_requises', [])
    print(f'  [intel] OK ({elapsed}s) — score={score} | corrections={corrections}')

    # Sauvegarder le cache
    with open(cache_path, 'w', encoding='utf-8') as f:
        json.dump(analysis, f, ensure_ascii=False, indent=2)

    intel_results[pid] = analysis

print(f'\n[INTEL] === Analyse terminee ({round(time.time()-t_intel_start,1)}s) ===')
print(f'[INTEL] {len(intel_results)}/{len(manifest["personnes"])} FBX analyses')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — GEMINI INJECTION (OPTIONNEL)
# Injecter intel_vision.json produit par Gemini Chat
#
# OPTION A : Deposer le fichier JSON dans TMP_DIR :
#   /content/tmp_ferrus/intel_vision.json
# puis executer cette cellule.
#
# OPTION B : Coller le JSON directement (decommenter INTEL_VISION_DATA)
#
# Format Gemini attendu pour instructions_camera :
#   { "suivre_personne_id": 2, "type_suivi": "static", "zoom_recommande": false }
#   NOTE : "actif" est absent — infere automatiquement si suivre_personne_id est defini.
#
# Si cette cellule n'est pas executee, camera_follow et mask_limbs
# seront desactives dans le plan de corrections.
# ═══════════════════════════════════════════════════════════════

# ── OPTION A : via fichier (GEMINI_IN sur Drive, ou TMP_DIR) ────
vision_path = os.path.join(GEMINI_IN, 'intel_vision.json')
if not os.path.exists(vision_path):
    vision_path = os.path.join(TMP_DIR, 'intel_vision.json')
if os.path.exists(vision_path):
    with open(vision_path, encoding='utf-8') as f:
        INTEL_VISION_DATA = json.load(f)
    print(f'[GEMINI] intel_vision.json charge depuis {vision_path}')
    nb_pers = len(INTEL_VISION_DATA.get('personnes', []))
    print(f'[GEMINI] Personnes detectees par Gemini : {nb_pers}')
    if 'instructions_camera' in INTEL_VISION_DATA:
        cam = INTEL_VISION_DATA['instructions_camera']
        # Gemini produit 'suivre_personne_id', pas 'cible_person_id' ni 'actif'
        cible = cam.get('suivre_personne_id', cam.get('cible_person_id', '?'))
        print(f'[GEMINI] Camera : cible=P{cible} | type={cam.get("type_suivi")}')
    for p in INTEL_VISION_DATA.get('personnes', []):
        hors = p.get('membres_hors_cadre', [])
        print(f'[GEMINI] P{p["person_id"]} : membres_hors_cadre = {hors if hors else "aucun"}')
else:
    INTEL_VISION_DATA = None
    print(f'[GEMINI] Pas de intel_vision.json dans GEMINI_IN/ ni TMP — camera_follow et mask_limbs desactives')

# ── OPTION B : coller le JSON ici ─────────────────────────────────
# INTEL_VISION_DATA = {
#     "scene_id": "video_01",
#     "video": {"duration_sec": 14.0, "fps_detected": 30},
#     "personnes": [
#         {"person_id": 1, "membres_hors_cadre": ["jambe_gauche", "jambe_droite"]},
#         {"person_id": 2, "membres_hors_cadre": ["jambe_gauche", "jambe_droite"]}
#     ],
#     "instructions_camera": {
#         "suivre_personne_id": 2,
#         "type_suivi": "static",
#         "zoom_recommande": false
#     }
# }


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — MERGE
# Fusionne intel_skeleton + intel_vision → plan_corrections.json
# Produit un plan global + un fichier plat par personne pour EXEC
# ═══════════════════════════════════════════════════════════════

def merge_intel(manifest, intel_results, intel_vision=None):
    """
    Construit plan_corrections.json a partir des analyses INTEL.

    Compatibilite champs Gemini :
      - instructions_camera.suivre_personne_id  (Gemini natif)
      - instructions_camera.cible_person_id     (fallback legacy)
      - instructions_camera.actif               (absent chez Gemini — infere)

    Args:
        manifest      : dict charge depuis manifest.json
        intel_results : {pid: analyse_dict} produit par CELL 5
        intel_vision  : dict intel_vision.json (Gemini) ou None

    Returns:
        plan global {version, personnes: [...]}
    """
    plan = {'version': '1.0', 'personnes': []}

    # ── Pre-calcul camera (hors boucle personnes) ─────────────────
    cam_global_on = False
    cam_cible     = None
    cam_type      = 'smooth_follow'
    if intel_vision:
        cam_cfg = intel_vision.get('instructions_camera', {})
        # Gemini utilise 'suivre_personne_id', fallback sur 'cible_person_id'
        cam_cible = cam_cfg.get('suivre_personne_id', cam_cfg.get('cible_person_id', None))
        cam_type  = cam_cfg.get('type_suivi', 'smooth_follow')
        # 'actif' absent chez Gemini : infere True si une cible est definie
        cam_global_on = cam_cfg.get('actif', cam_cible is not None)

    for person in manifest['personnes']:
        pid     = person['person_id']
        pid_int = int(pid[1:])   # 'P1' -> 1, 'P2' -> 2
        fbx_in  = person['fbx_input']
        fbx_out = os.path.join(OUT_DIR, f'ferrus_{pid}.fbx')

        skel    = intel_results.get(pid, {})
        req     = skel.get('corrections_requises', [])
        qualite = skel.get('qualite_fcurves', {})

        # ── smooth_fcurves ──────────────────────────────────────
        smooth_on    = 'smooth_fcurves' in req
        jitter_bones = qualite.get('jitter_bones', [])
        if smooth_on and not jitter_bones:
            jitter_bones = ['l_hand_JNT', 'r_hand_JNT', 'head_JNT']  # fallback

        # ── stabilize_hips ──────────────────────────────────────
        hips_on   = 'stabilize_hips' in req
        derive    = qualite.get('derive_hanches', {})
        derive_cm = derive.get('delta_vertical_cm', 5.0)
        direction = derive.get('direction', 'descente_progressive')

        # ── remove_foot_slide ───────────────────────────────────
        # Desactive si les jambes sont hors cadre (inutile de corriger ce qu'on ne voit pas)
        foot_on = 'remove_foot_slide' in req

        # ── camera_follow ────────────────────────────────────────
        # Camera uniquement pour la personne cible
        cam_actif_person = cam_global_on and (cam_cible == pid_int)

        # ── mask_limbs ───────────────────────────────────────────
        # Chaque personne a ses propres membres hors cadre
        mask_actif   = False
        membres_mask = []
        if intel_vision:
            for p in intel_vision.get('personnes', []):
                if p.get('person_id') == pid_int:
                    hors_cadre = p.get('membres_hors_cadre', [])
                    if hors_cadre:
                        mask_actif   = True
                        membres_mask = hors_cadre
                    # Si les jambes sont hors cadre, remove_foot_slide est inutile
                    if any('jambe' in m or 'pied' in m for m in hors_cadre):
                        foot_on = False
                    break

        person_plan = {
            'person_id': pid,
            'fbx_input': fbx_in,
            'fbx_output': fbx_out,
            'corrections_exec': {
                'smooth_fcurves': {
                    'enabled':            smooth_on,
                    'bones_cibles':       jitter_bones,
                    'intensite':          0.5,
                    'gimbal_lock_risque': qualite.get('gimbal_lock_risque', False),
                },
                'stabilize_hips': {
                    'enabled':                 hips_on,
                    'correction_verticale_cm': derive_cm,
                    'direction':               direction,
                },
                'remove_foot_slide': {
                    'enabled':  foot_on,
                    'seuil_cm': 2.0,
                },
                'camera_follow': {
                    'actif':           cam_actif_person,
                    'cible_person_id': cam_cible,
                    'type_suivi':      cam_type,
                },
                'mask_limbs': {
                    'actif':             mask_actif,
                    'membres_a_masquer': membres_mask,
                },
            },
            'retargeting_r15': {
                'enabled':          True,
                't_pose_disponible': skel.get('metadata', {}).get('t_pose_incluse', False),
            },
        }
        plan['personnes'].append(person_plan)

    return plan


# ── Execution du merge ────────────────────────────────────────────
assert intel_results, 'Executer CELL 5 - INTEL en premier'

_vision = globals().get('INTEL_VISION_DATA', None)
plan    = merge_intel(manifest, intel_results, intel_vision=_vision)

# Sauvegarder plan_corrections.json global
plan_global_path = os.path.join(TMP_DIR, 'plan_corrections.json')
with open(plan_global_path, 'w', encoding='utf-8') as f:
    json.dump(plan, f, ensure_ascii=False, indent=2)

# ── Affichage du plan ─────────────────────────────────────────────
print('[MERGE] Plan de corrections genere :')
print(f'  Fichier : {plan_global_path}')
print()
for pp in plan['personnes']:
    pid = pp['person_id']
    ce  = pp['corrections_exec']
    print(f'  {pid} : {os.path.basename(pp["fbx_input"])}')
    print(f'    smooth_fcurves   : {ce["smooth_fcurves"]["enabled"]}'
          f' | bones : {ce["smooth_fcurves"]["bones_cibles"]}')
    print(f'    stabilize_hips   : {ce["stabilize_hips"]["enabled"]}'
          f' | {ce["stabilize_hips"]["direction"]} ({ce["stabilize_hips"]["correction_verticale_cm"]} cm)')
    print(f'    remove_foot_slide: {ce["remove_foot_slide"]["enabled"]}')
    print(f'    camera_follow    : actif={ce["camera_follow"]["actif"]}'
          f' | cible=P{ce["camera_follow"]["cible_person_id"]} | type={ce["camera_follow"]["type_suivi"]}')
    print(f'    mask_limbs       : actif={ce["mask_limbs"]["actif"]}'
          f' | membres={ce["mask_limbs"]["membres_a_masquer"]}')
    print()

print('[MERGE] Validez le plan ci-dessus avant de lancer CELL 8 - EXEC')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — EXEC
# Execute les 5 operations de correction via Blender headless
# Ordre : smooth → hips → foot → mask → camera
# Chaque operation prend le FBX de sortie de la precedente
# ═══════════════════════════════════════════════════════════════
assert plan, 'Executer CELL 7 - MERGE en premier'

EXEC_OPS = [
    ('smooth_fcurves',    os.path.join(EXEC_DIR, 'smooth_fcurves.py')),
    ('stabilize_hips',    os.path.join(EXEC_DIR, 'stabilize_hips.py')),
    ('remove_foot_slide', os.path.join(EXEC_DIR, 'remove_foot_slide.py')),
    ('mask_limbs',        os.path.join(EXEC_DIR, 'mask_limbs.py')),
    ('camera_follow',     os.path.join(EXEC_DIR, 'camera_follow.py')),
]


def _run_blender_op(script_path, fbx_in, plan_path, fbx_out, op_name):
    """Lance un script EXEC via Blender headless. Retourne le rapport ou False."""
    t0  = time.time()
    cmd = [
        BLENDER_BIN, '--background', '--python', script_path,
        '--', '--fbx-in', fbx_in, '--plan', plan_path, '--fbx-out', fbx_out
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    elapsed = round(time.time() - t0, 1)

    if proc.returncode != 0:
        print(f'  [{op_name}] ERREUR code={proc.returncode} ({elapsed}s)')
        if proc.stderr:
            print(proc.stderr[-600:])
        return False

    # Parser le rapport JSON dans stdout
    rapport = {'status': 'ok'}
    try:
        lines     = proc.stdout.split('\n')
        json_idx  = next((i for i, l in enumerate(lines) if l.strip().startswith('{')), None)
        if json_idx is not None:
            rapport = json.loads('\n'.join(lines[json_idx:]).strip())
    except Exception:
        pass

    status = rapport.get('status', 'ok')
    print(f'  [{op_name}] {status.upper()} ({elapsed}s)')
    return rapport


exec_rapports = {}
t_exec_start  = time.time()

for person_plan in plan['personnes']:
    pid      = person_plan['person_id']
    fbx_orig = person_plan['fbx_input']

    # Plan plat par personne (contrat attendu par les scripts EXEC)
    flat_plan = {
        'corrections_exec': person_plan['corrections_exec'],
        'retargeting_r15':  person_plan['retargeting_r15'],
    }
    flat_plan_path = os.path.join(TMP_DIR, f'plan_{pid}.json')
    with open(flat_plan_path, 'w', encoding='utf-8') as f:
        json.dump(flat_plan, f, ensure_ascii=False, indent=2)

    print(f'\n[EXEC] ═══ {pid} : {os.path.basename(fbx_orig)} ═══')

    current_fbx      = fbx_orig
    person_rapports  = {}
    exec_ok          = True

    for op_name, script_path in EXEC_OPS:
        out_fbx = os.path.join(TMP_DIR, f'{pid}_{op_name}.fbx')
        rapport = _run_blender_op(script_path, current_fbx, flat_plan_path, out_fbx, op_name)
        person_rapports[op_name] = rapport

        if rapport is False:
            exec_ok = False
            print(f'  Arret de la chaine EXEC sur {op_name}')
            break

        # Le FBX de sortie existe si l'op a modifie OU copie le fichier
        if os.path.exists(out_fbx):
            current_fbx = out_fbx

    exec_rapports[pid] = {
        'fbx_exec_final': current_fbx,
        'operations':     person_rapports,
        'status':         'ok' if exec_ok else 'erreur',
    }
    print(f'  FBX EXEC final : {os.path.basename(current_fbx)}')

print(f'\n[EXEC] === Compartiment EXEC TERMINE ({round(time.time()-t_exec_start,1)}s) ===')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — OUTPUT
# Retargeting DeepMotion → Roblox R15 via Blender headless
# Produit les FBX R15 finaux dans OUT/
# ═══════════════════════════════════════════════════════════════
assert exec_rapports, 'Executer CELL 8 - EXEC en premier'

RETARGET_SCRIPT = os.path.join(OUTPUT_DIR, 'retarget_r15.py')
output_rapports = {}
t_out_start     = time.time()

for person_plan in plan['personnes']:
    pid         = person_plan['person_id']
    fbx_exec    = exec_rapports[pid]['fbx_exec_final']
    fbx_r15_out = person_plan['fbx_output']  # OUT/ferrus_P1.fbx
    flat_plan_path = os.path.join(TMP_DIR, f'plan_{pid}.json')

    print(f'\n[OUTPUT] ═══ {pid} ═══')
    print(f'  FBX exec  : {os.path.basename(fbx_exec)}')
    print(f'  FBX R15   : {os.path.basename(fbx_r15_out)}')

    t0  = time.time()
    cmd = [
        BLENDER_BIN, '--background', '--python', RETARGET_SCRIPT,
        '--', '--fbx-in', fbx_exec, '--plan', flat_plan_path, '--fbx-out', fbx_r15_out
    ]
    proc    = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
    elapsed = round(time.time() - t0, 1)

    if proc.returncode != 0:
        print(f'  ERREUR code={proc.returncode} ({elapsed}s)')
        print(proc.stderr[-800:] if proc.stderr else '(pas de stderr)')
        output_rapports[pid] = {'status': 'erreur', 'fbx_r15': None, 'duree_s': elapsed}
        continue

    # Parser le rapport JSON dans stdout
    rapport = {'status': 'ok'}
    try:
        lines    = proc.stdout.split('\n')
        json_idx = next((i for i, l in enumerate(lines) if l.strip().startswith('{')), None)
        if json_idx is not None:
            rapport = json.loads('\n'.join(lines[json_idx:]).strip())
    except Exception:
        pass

    frames = rapport.get('frames_transferees', '?')
    bones  = len(rapport.get('bones_retargetes', []))
    print(f'  OK ({elapsed}s) — {frames} frames | {bones}/15 bones')
    output_rapports[pid] = {
        'status':  'ok',
        'fbx_r15': fbx_r15_out,
        'duree_s': elapsed,
        'rapport': rapport,
    }

print(f'\n[OUTPUT] === Compartiment OUTPUT TERMINE ({round(time.time()-t_out_start,1)}s) ===')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 10 — RAPPORT
# Genere rapport.json et affiche le bilan final
# ═══════════════════════════════════════════════════════════════
import datetime
assert output_rapports, 'Executer CELL 9 - OUTPUT en premier'

rapport_global = {
    'ferrus_version':    '1.0',
    'date_traitement':   datetime.datetime.now().isoformat(),
    'nb_personnes':      len(plan['personnes']),
    'personnes':         [],
}

print('=' * 65)
print('  FERRUS ANIMUS — RAPPORT FINAL')
print('=' * 65)

all_ok = True

for person_plan in plan['personnes']:
    pid       = person_plan['person_id']
    skel      = intel_results.get(pid, {})
    score     = skel.get('qualite_fcurves', {}).get('score_global', 'N/A')
    req       = skel.get('corrections_requises', [])
    r15_valid = skel.get('squelette', {}).get('mapping_r15_valide', '?')
    out_rpt   = output_rapports.get(pid, {})
    status    = out_rpt.get('status', 'inconnu')
    fbx_r15   = out_rpt.get('fbx_r15', None)
    duree     = out_rpt.get('duree_s', 0)
    exec_rpt  = exec_rapports.get(pid, {})

    if status != 'ok':
        all_ok = False

    print(f'\n  {pid}  {os.path.basename(person_plan["fbx_input"])}')
    print(f'    Score qualite input   : {score}')
    print(f'    Corrections requises  : {req if req else "aucune"}')
    print(f'    Mapping R15 valide    : {r15_valid}')

    # Detail EXEC
    ops_applied = []
    for op_name, op_rpt in exec_rpt.get('operations', {}).items():
        if op_rpt and op_rpt.get('status') == 'ok':
            ops_applied.append(op_name)
    print(f'    Operations EXEC ok    : {ops_applied if ops_applied else "aucune"}')

    print(f'    Status output         : {status.upper()}')
    print(f'    FBX R15               : {os.path.basename(fbx_r15) if fbx_r15 else "N/A"}')
    print(f'    Duree retargeting     : {duree}s')

    rapport_global['personnes'].append({
        'person_id':              pid,
        'fbx_input':              person_plan['fbx_input'],
        'fbx_r15':                fbx_r15,
        'score_qualite_input':    score,
        'corrections_requises':   req,
        'operations_exec_ok':     ops_applied,
        'mapping_r15_valide':     r15_valid,
        'status':                 status,
        'duree_retargeting_s':    duree,
    })

# Sauvegarder rapport.json
rapport_path = os.path.join(OUT_DIR, 'rapport.json')
with open(rapport_path, 'w', encoding='utf-8') as f:
    json.dump(rapport_global, f, ensure_ascii=False, indent=2)

print()
print('=' * 65)
print(f'  rapport.json : {rapport_path}')
print(f'  FBX R15      : {OUT_DIR}/')
print('=' * 65)
if all_ok:
    print()
    print('  VICTOIRE. LA FLOTTE FERRUS A TRIOMPHE.')
    print('  POUR L\'EMPEROR. POUR L\'IMPERIUM.')
else:
    print()
    print('  CERTAINES PERSONNES ONT ECHOUE. CONSULTER LE RAPPORT.')
print('=' * 65)